# Piyu AI Fashion Design Generator — CLEAN Colab

**Pipeline:** RealVisXL → SAM → IDM-VTON

This notebook is rebuilt to remove the dependency and import problems from the previous notebook:

- No `auto1111sdk`
- No `requirements.txt` installation
- No repeated/conflicting Torch installations
- Fixed Torch/Torchvision compatibility
- Fixed Pillow `_Ink` error
- Fixed NumPy/OpenCV compatibility
- Fixed `accelerate`/`peft` mismatch
- Fixed `transformers`/`diffusers` mismatch
- Fixed IDM-VTON `src` import path
- Uses Diffusers `from_single_file()` for the local RealVisXL checkpoint
- Releases GPU memory between RealVisXL and SAM

Start with a fresh Colab runtime and run the cells from top to bottom.

## STEP 0 — Fresh runtime

Select:

**Runtime → Disconnect and delete runtime → Reconnect**

Do not run the old notebook's installation cells.

In [ ]:
# STEP 1 — CONTROLLED DEPENDENCY SETUP

import sys, subprocess, importlib.metadata as md

def ver(name):
    try:
        return md.version(name)
    except md.PackageNotFoundError:
        return None

torch_v = ver("torch")
tv_v = ver("torchvision")

print("Python:", sys.version.split()[0])
print("Existing Torch:", torch_v)
print("Existing Torchvision:", tv_v)

packages = [
    "numpy==1.26.4",
    "Pillow==10.4.0",
    "opencv-python==4.10.0.84",
    "scipy==1.13.1",
    "diffusers==0.27.2",
    "transformers==4.37.2",
    "accelerate==0.30.1",
    "peft==0.10.0",
    "huggingface_hub==0.23.4",
    "tokenizers==0.15.2",
    "safetensors==0.4.3",
    "einops==0.8.0",
    "timm==0.9.16",
    "onnxruntime==1.20.1",
    "pycocotools",
    "fvcore",
    "cloudpickle",
    "av",
    "tqdm",
]

print("\nInstalling controlled project dependencies...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--no-cache-dir", "--upgrade", *packages],
    check=True
)

print("\nInstalling Segment Anything...")
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "--no-cache-dir",
        "git+https://github.com/facebookresearch/segment-anything.git"
    ],
    check=True
)

if not (torch_v == "2.11.0" and tv_v == "0.26.0"):
    print("\nTorch/Torchvision mismatch detected.")
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install",
            "--no-cache-dir", "--force-reinstall",
            "torch==2.11.0",
            "torchvision==0.26.0",
            "torchaudio==2.11.0",
            "--index-url", "https://download.pytorch.org/whl/cu128"
        ],
        check=True
    )
    print("\n⚠️ Torch was changed. Restart the Colab runtime now.")
else:
    print("\n✅ Existing Torch/Torchvision pair is correct.")

print("\nSTEP 1 complete.")

In [ ]:
# STEP 2 — VERIFY THE EXACT FAILURE POINTS

import sys
import numpy as np
import PIL
import torch
import torchvision
import diffusers
import transformers
import accelerate
import peft
import huggingface_hub

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("Pillow:", PIL.__version__)
print("Torch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("Diffusers:", diffusers.__version__)
print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)
print("PEFT:", peft.__version__)
print("HuggingFace Hub:", huggingface_hub.__version__)

assert np.__version__.startswith("1.26."), "Wrong NumPy"
assert PIL.__version__.startswith("10.4."), "Wrong Pillow"
assert torch.__version__.startswith("2.11."), "Wrong Torch"
assert torchvision.__version__.startswith("0.26."), "Wrong Torchvision"
assert diffusers.__version__ == "0.27.2", "Wrong Diffusers"
assert transformers.__version__ == "4.37.2", "Wrong Transformers"
assert accelerate.__version__ == "0.30.1", "Wrong Accelerate"
assert peft.__version__ == "0.10.0", "Wrong PEFT"

assert torch.cuda.is_available(), "GPU is not enabled."
print("GPU:", torch.cuda.get_device_name(0))

# Previous torchvision::nms failure
boxes = torch.tensor([[0, 0, 100, 100]], dtype=torch.float32)
scores = torch.tensor([0.9])
print("NMS:", torchvision.ops.nms(boxes, scores, 0.5))
print("✅ Torchvision NMS works.")

from segment_anything import SamPredictor, sam_model_registry
print("✅ SAM import works.")

from transformers import CLIPImageProcessor
print("✅ CLIPImageProcessor works.")

from diffusers import StableDiffusionXLPipeline
print("✅ StableDiffusionXLPipeline works.")

print("\n🎉 DEPENDENCY CHECK PASSED")

In [ ]:
# STEP 3 — CLONE PROJECT + IDM-VTON

from pathlib import Path
import subprocess, os, sys

REPO_URL = "https://github.com/Piyu242005/AI-Fashion-Design-Generator-IBM-INTERNSHIP-2026.git"
# Correct repository URL:
REPO_URL = "https://github.com/Piyu242005/AI-Fashion-Design-Generator-IBM-INTERSHIP-2026.git"

REPO_ROOT = Path("/content/AI-Fashion-Design-Generator-IBM-INTERSHIP-2026")
PROJECT = REPO_ROOT / "Piyu-AI-Clothing-Fashion-Design-Generator"
IDM = PROJECT / "idm_vton"

if not PROJECT.exists():
    if not REPO_ROOT.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
    else:
        raise RuntimeError("Repository exists but project folder is missing.")

if not IDM.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/yisol/IDM-VTON.git", str(IDM)],
        check=True
    )

for d in [
    PROJECT / "weights",
    PROJECT / "reference_images",
    PROJECT / "samples",
    PROJECT / "results",
    IDM / "ckpt/densepose",
    IDM / "ckpt/humanparsing",
    IDM / "ckpt/openpose/ckpts",
]:
    d.mkdir(parents=True, exist_ok=True)

os.chdir(PROJECT)

if str(IDM) not in sys.path:
    sys.path.insert(0, str(IDM))

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("PROJECT:", PROJECT)
print("IDM-VTON:", IDM)
print("src exists:", (IDM / "src").exists())
print("Working directory:", Path.cwd())

In [ ]:
# STEP 4A — FIND OR DOWNLOAD REALVISXL + SAM

from pathlib import Path
import os, shutil, subprocess

def find_first(filename):
    for p in Path("/content").rglob(filename):
        if p.is_file():
            return p
    return None

def link_or_copy(src, dst):
    src, dst = Path(src), Path(dst)
    if dst.exists():
        return
    try:
        os.symlink(src.resolve(), dst)
        print("Linked:", dst)
    except OSError:
        shutil.copy2(src, dst)
        print("Copied:", dst)

realvis_dst = PROJECT / "weights/realvisxl.safetensors"
realvis_src = find_first("realvisxl.safetensors")

if realvis_src:
    print("Found RealVisXL:", realvis_src)
    link_or_copy(realvis_src, realvis_dst)
else:
    print("Downloading RealVisXL...")
    subprocess.run([
        "wget", "-c", "-O", str(realvis_dst),
        "https://civitai.com/api/download/models/361593?type=Model&format=SafeTensor&size=pruned&fp=fp16"
    ], check=True)

sam_dst = PROJECT / "weights/sam_vit_h_4b8939.pth"
sam_src = find_first("sam_vit_h_4b8939.pth")

if sam_src:
    print("Found SAM:", sam_src)
    link_or_copy(sam_src, sam_dst)
else:
    print("Downloading SAM ViT-H...")
    subprocess.run([
        "wget", "-c", "-O", str(sam_dst),
        "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth"
    ], check=True)

print("RealVisXL:", round(realvis_dst.stat().st_size / 1024**3, 2), "GB")
print("SAM:", round(sam_dst.stat().st_size / 1024**3, 2), "GB")

In [ ]:
# STEP 4B — IDM-VTON SUPPORT CHECKPOINTS

from pathlib import Path
import subprocess

downloads = [
    (
        IDM / "ckpt/densepose/model_final_162be9.pkl",
        "https://huggingface.co/spaces/yisol/IDM-VTON/resolve/main/ckpt/densepose/model_final_162be9.pkl?download=true"
    ),
    (
        IDM / "ckpt/humanparsing/parsing_atr.onnx",
        "https://huggingface.co/spaces/yisol/IDM-VTON/resolve/main/ckpt/humanparsing/parsing_atr.onnx?download=true"
    ),
    (
        IDM / "ckpt/humanparsing/parsing_lip.onnx",
        "https://huggingface.co/spaces/yisol/IDM-VTON/resolve/main/ckpt/humanparsing/parsing_lip.onnx?download=true"
    ),
    (
        IDM / "ckpt/openpose/ckpts/body_pose_model.pth",
        "https://huggingface.co/spaces/yisol/IDM-VTON/resolve/main/ckpt/openpose/ckpts/body_pose_model.pth?download=true"
    ),
]

for dst, url in downloads:
    if dst.exists() and dst.stat().st_size > 0:
        print("Already exists:", dst)
    else:
        print("Downloading:", dst)
        subprocess.run(["wget", "-c", "-O", str(dst), url], check=True)

print("\n✅ IDM-VTON support checkpoints ready.")

## STEP 5 — RealVisXL loader fix

The old project used `auto1111sdk`. That package is removed from this notebook.

Diffusers officially supports loading SDXL `.safetensors` checkpoints with `StableDiffusionXLPipeline.from_single_file()`. citeturn0search0turn0search10

In [ ]:
# STEP 5 — CREATE A CLEAN REALVISXL GENERATOR

from pathlib import Path

generator_code = r"""
import argparse
import gc
import random
from pathlib import Path

import torch
from diffusers import StableDiffusionXLPipeline, DPMSolverSinglestepScheduler

MODEL_PATH = "weights/realvisxl.safetensors"

def load_pipeline():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Loading RealVisXL:", MODEL_PATH)

    if not Path(MODEL_PATH).exists():
        raise FileNotFoundError(MODEL_PATH)

    pipe = StableDiffusionXLPipeline.from_single_file(
        MODEL_PATH,
        config="stabilityai/stable-diffusion-xl-base-1.0",
        torch_dtype=torch.float16,
        use_safetensors=True,
        safety_checker=None,
    )

    try:
        pipe.scheduler = DPMSolverSinglestepScheduler.from_config(
            pipe.scheduler.config,
            use_karras_sigmas=True,
        )
    except Exception:
        pass

    pipe.enable_vae_slicing()
    pipe.enable_vae_tiling()
    pipe.enable_model_cpu_offload()

    return pipe

def generate(prompt, output_path, steps=4, width=768, height=1024):
    pipe = load_pipeline()

    negative = (
        "low quality, blurry, deformed, bad anatomy, bad hands, "
        "extra fingers, fused fingers, distorted face, duplicate person, "
        "cropped head, text, watermark, cartoon, illustration"
    )

    seed = random.randint(0, 2**32 - 1)
    generator = torch.Generator(device="cpu").manual_seed(seed)

    print("Seed:", seed)
    print("Prompt:", prompt)

    image = pipe(
        prompt=prompt,
        negative_prompt=negative,
        width=width,
        height=height,
        num_inference_steps=steps,
        guidance_scale=0.0,
        generator=generator,
    ).images[0]

    output = Path(output_path)
    output.parent.mkdir(parents=True, exist_ok=True)
    image.save(output)

    print("Saved:", output)

    del pipe
    gc.collect()
    torch.cuda.empty_cache()

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--prompt", required=True)
    parser.add_argument("--output_path", required=True)
    parser.add_argument("--steps", type=int, default=4)
    parser.add_argument("--width", type=int, default=768)
    parser.add_argument("--height", type=int, default=1024)
    args = parser.parse_args()

    generate(
        args.prompt,
        args.output_path,
        args.steps,
        args.width,
        args.height,
    )
"""

(PROJECT / "generate_model_colab.py").write_text(generator_code)
print("✅ Created generate_model_colab.py")

In [ ]:
# STEP 6 — GENERATE AI FASHION MODEL

import subprocess
from pathlib import Path

MODEL_IMAGE = "reference_images/fashion_model.png"

PROMPT = (
    "a modern professional fashion model, full body portrait, "
    "wearing a modern black crop top and high waist skirt, "
    "realistic human proportions, natural skin texture, "
    "studio fashion photography, neutral studio background, "
    "high detail clothing, realistic fabric, photorealistic"
)

subprocess.run([
    sys.executable,
    "generate_model_colab.py",
    "--prompt", PROMPT,
    "--output_path", MODEL_IMAGE,
    "--steps", "4",
    "--width", "768",
    "--height", "1024",
], check=True)

assert Path(MODEL_IMAGE).exists()
print("✅ Generated:", MODEL_IMAGE)

In [ ]:
# STEP 7 — DISPLAY MODEL

from IPython.display import display
from PIL import Image

display(Image.open(MODEL_IMAGE).convert("RGB"))

In [ ]:
# STEP 8A — DISPLAY MODEL WITH COORDINATES

import matplotlib.pyplot as plt
from PIL import Image

img = Image.open(MODEL_IMAGE).convert("RGB")

print("Image size:", img.size)

plt.figure(figsize=(9, 12))
plt.imshow(img)
plt.xlim(0, img.width)
plt.ylim(img.height, 0)
plt.xticks(range(0, img.width + 1, 100))
plt.yticks(range(0, img.height + 1, 100))
plt.grid()
plt.title("Choose 3 points INSIDE the clothing")
plt.show()

In [ ]:
# STEP 8B — ENTER THREE CLOTHING POINTS

points = []

for i in range(3):
    while True:
        raw = input(f"Point {i+1} (x,y): ").strip()
        try:
            x, y = [int(v.strip()) for v in raw.split(",")]
            if 0 <= x < img.width and 0 <= y < img.height:
                points.append([x, y])
                break
            print("❌ Point is outside the image.")
        except Exception:
            print("❌ Format must be like: 380,420")

print("Selected points:", points)

In [ ]:
# STEP 8C — RUN SAM

import gc
import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image
from segment_anything import SamPredictor, sam_model_registry

SAM_PATH = str(PROJECT / "weights/sam_vit_h_4b8939.pth")
MASK_PATH = "reference_images/fashion_model_mask.png"

sam = sam_model_registry["vit_h"](checkpoint=SAM_PATH)
sam.to(device="cuda")

predictor = SamPredictor(sam)

rgb = np.array(Image.open(MODEL_IMAGE).convert("RGB"))
predictor.set_image(rgb)

input_point = np.array(points, dtype=np.float32)
input_label = np.ones(len(points), dtype=np.int32)

masks, scores, _ = predictor.predict(
    point_coords=input_point,
    point_labels=input_label,
    multimask_output=True,
)

best = int(np.argmax(scores))
mask = masks[best]

cv2.imwrite(
    MASK_PATH,
    (mask.astype(np.uint8) * 255)
)

print("Mask:", MASK_PATH)
print("SAM score:", float(scores[best]))

plt.figure(figsize=(9, 12))
plt.imshow(rgb)
plt.imshow(mask, alpha=0.45)
plt.scatter(input_point[:, 0], input_point[:, 1], c="red", s=80)
plt.axis("off")
plt.show()

del predictor, sam
gc.collect()
torch.cuda.empty_cache()

print("✅ SAM released from GPU.")

In [ ]:
# STEP 9 — UPLOAD GARMENT

from google.colab import files
from shutil import copyfile
from PIL import Image
from IPython.display import display

uploaded = files.upload()

if not uploaded:
    raise RuntimeError("No garment image uploaded.")

uploaded_name = next(iter(uploaded))
GARMENT_PATH = "samples/garment.png"

copyfile(uploaded_name, GARMENT_PATH)

garment = Image.open(GARMENT_PATH).convert("RGB")
print("Garment size:", garment.size)
display(garment)
print("✅ Saved:", GARMENT_PATH)

## STEP 10 — IDM-VTON import

The official IDM-VTON implementation is available from the project's GitHub repository, and the official Hugging Face model is `yisol/IDM-VTON`. citeturn3search3turn4view0

The previous `ModuleNotFoundError: No module named 'src'` happened because `try_on.py` was launched without putting the `idm_vton` directory on the subprocess `PYTHONPATH`.

In [ ]:
# STEP 10 — VERIFY IDM-VTON SRC

import os, sys
from pathlib import Path

os.chdir(PROJECT)

if str(IDM) not in sys.path:
    sys.path.insert(0, str(IDM))

assert (IDM / "src").exists(), "idm_vton/src is missing"
assert Path(MODEL_IMAGE).exists(), "Model image missing"
assert Path(MASK_PATH).exists(), "Mask missing"
assert Path(GARMENT_PATH).exists(), "Garment missing"
assert Path("try_on.py").exists(), "try_on.py missing"

from src.tryon_pipeline import StableDiffusionXLInpaintPipeline

print("✅ IDM-VTON src import works.")

In [ ]:
# STEP 11 — RUN IDM-VTON

import os
import subprocess
from pathlib import Path

CLOTH_TYPE = "jacket"
FINAL_IMAGE = "results/final_tryon.png"

env = os.environ.copy()
env["PYTHONPATH"] = str(IDM) + os.pathsep + env.get("PYTHONPATH", "")
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
env["TOKENIZERS_PARALLELISM"] = "false"

cmd = [
    sys.executable,
    "try_on.py",
    "--reference_image", MODEL_IMAGE,
    "--mask", MASK_PATH,
    "--garment", GARMENT_PATH,
    "--cloth_type", CLOTH_TYPE,
    "--output_path", FINAL_IMAGE,
]

print("Running IDM-VTON...")
print(" ".join(cmd))

subprocess.run(
    cmd,
    cwd=str(PROJECT),
    env=env,
    check=True
)

if not Path(FINAL_IMAGE).exists():
    raise RuntimeError("IDM-VTON did not create the final image.")

print("✅ Final image:", FINAL_IMAGE)

In [ ]:
# STEP 12 — DISPLAY FINAL RESULT

from IPython.display import display
from PIL import Image
from pathlib import Path

result = Image.open(FINAL_IMAGE).convert("RGB")
print("FINAL RESULT:", result.size)
display(result)

# ✅ COMPLETE

Expected files:

```text
reference_images/fashion_model.png
reference_images/fashion_model_mask.png
samples/garment.png
results/final_tryon.png
```

### Main fixes compared with the previous notebook

| Previous problem | Fixed |
|---|---|
| `auto1111sdk` dependency conflict | Removed |
| `operator torchvision::nms does not exist` | Matching Torch/Torchvision |
| `PIL._typing._Ink` | Pillow pinned to 10.4.0 |
| NumPy 1.x / 2.x chaos | NumPy pinned to 1.26.4 |
| `clear_device_cache` | Accelerate + PEFT pinned together |
| `CLIPImageProcessor` import failure | Transformers pinned |
| `ModuleNotFoundError: src` | Explicit IDM-VTON `PYTHONPATH` |
| Old A1111 RealVisXL loader | Diffusers `from_single_file()` |
| GPU memory overlap | RealVisXL and SAM released between stages |

**Note:** no notebook can guarantee zero errors from a temporary Colab runtime or external model download, but this version removes the specific dependency and import conflicts shown in your uploaded notebook and uses one consistent environment.